In [1]:
# Install all libraries by running in the terminal: pip install -r requirements.txt
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from docx import Document
import PyPDF2
import os
import tempfile
import tiktoken
import pinecone
from langchain_community.vectorstores import Pinecone
from pinecone import ServerlessSpec
from langchain_community.document_loaders import AmazonTextractPDFLoader

import boto3
from botocore.exceptions import ClientError


/Users/alexanderarefolov/Dropbox/Coding_Projects/knowledge_retrieval_LLM_chatbot_Streamlit_app/knowledge_retrieval_LLM_chatbot_Streamlit_app/.venv/lib/python3.11/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# fetch environmental variables
load_dotenv()

True

### Helper Functions

In [3]:
# create s3 bucket if does not exist
def create_bucket(bucket_name, region = os.getenv("AWS_REGION")):
    """Create an S3 bucket in a specified region if
    bucket does not already exist.

    Parameters
    ----------

    If a region is not specified, the bucket is created in the S3 default
    region (us-east-1).

    :param bucket_name: Bucket to create
    :param region: String region to create bucket in, e.g., 'us-west-2'
    :return: True if bucket created, else False
    """
    try:
        s3_client = boto3.client('s3', region_name = region)

        response = s3_client.list_buckets()

        if bucket_name in response["Buckets"]:
            print(f'Bucket {bucket_name} already exists in region {region}')
        else:
            location = {'LocationConstraint': region}
            s3_client.create_bucket(
                Bucket=bucket_name, 
                CreateBucketConfiguration=location
                )
    except ClientError as e:
        print(e)
        return False
    return True


In [4]:
# upload fole to s3
def upload_file_s3(file_name, bucket, namespace, s3_file_name):
    """Upload a file to an S3 bucket
    
    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param namespace: name for the current session for a given user. Same
    namespace name will be used to create namespace in Pinecone index
    :param s3_file_name: name of the file in S3
    :return: True if file was uploaded, else False
    """
    object_name = f'{namespace}/{s3_file_name}'

    # Upload the file
    s3_client = boto3.client('s3')
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        print(e)
        return False
    print(f"file {file_name} has been uploaded into the bucket {bucket}.")
    return True

In [5]:
# delete file from s3
def delete_file_from_s3(bucket, namespace, s3_file_name):
    """Delete file from S3 bucket."""

    key = f"{namespace}/{s3_file_name}"

    # Create S3 client
    s3 = boto3.client('s3')

    # Delete file from S3
    s3.delete_object(Bucket = bucket, Key = key)
    print(f"File {key} deleted from S3 bucket {bucket}.")

    return True

In [6]:
# get s3 url of the file uploaded to s3
def get_s3_url(bucket, namespace, s3_file_name):

    s3_uri = "s3://" + bucket + "/" + namespace + "/" + s3_file_name

    return s3_uri



In [7]:
# load document into Amazon Textract
def load_document_AWS_textract(s3_url):

    textract_client = boto3.client("textract", region_name="us-east-2")

    loader = AmazonTextractPDFLoader(file_path = s3_url, client=textract_client)

    documents = loader.load()

    print(f"Loaded documents {len(documents)} pages long")

    return documents

In [8]:
# splitting data in chunks
def chunk_data(data, chunk_size = 1024, chunk_overlap = 80):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap)
    chunks = text_splitter.split_documents(data)
    if len(chunks) == 0:
        raise ValueError("Chunking failed - returned zero chunks!")
    return chunks

In [9]:
# calculate embedding cost using tiktoken
def calculate_input_embedding_cost(texts):
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    # print(f'Total Tokens: {total_tokens}')
    # print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')
    return total_tokens, (total_tokens / 1000000) * 0.02

In [10]:
def create_embeddings(chunks, namespace, index_name = os.getenv("PINECONE_INDEX_NAME")):

    pc = pinecone.Pinecone()
        
    embeddings = OpenAIEmbeddings(
        model = os.getenv("TEXT_EMBEDDING_MODEL"), 
        dimensions = os.getenv("TEXT_EMBEDDING_DIMENSIONS")
        )  # 512 works as well

    # create index if does not exist yet
    if index_name not in pc.list_indexes().names():
        print(f'Creating {index_name} index')
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec=ServerlessSpec(
                cloud='aws', 
                region='us-east-2'
            )
        )

    # processing the input documents, generating embeddings using the provided `OpenAIEmbeddings` instance,
    # inserting the embeddings into the index and returning a new Pinecone vector store object. 
    vector_store = Pinecone.from_documents(
        documents = chunks, 
        embedding = embeddings, 
        index_name = index_name, 
        namespace = namespace) 
    # processing the input documents, the chunks, geenrating the embeddings
    # using the provided openAI embeddings instance, inserting the embedding intot he index and returning pincone vectoor store object.
    print(f'Created vector store within {index_name} index and in {namespace} namespace')
        
    return vector_store

### Input Variables

In [11]:
pwd

'/Users/alexanderarefolov/Dropbox/Coding_Projects/knowledge_retrieval_LLM_chatbot_Streamlit_app/knowledge_retrieval_LLM_chatbot_Streamlit_app'

In [12]:
ls

Amendment_Master_Deed_scanned.pdf       app_scratch_book.ipynb
CD_seller_signed.pdf                    app_scratch_pinecone.ipynb
Hatching_a_story.docx                   final_signed_offer.pdf
LICENSE                                 full_code.ipynb
NLP_orig_text.pdf                       image2.png
Purchase_and_sale_final.pdf             image3.png
README.md                               image5.png
RSM_packing_list.docx                   image6.png
RSM_packing_list_2.docx                 img.png
Software_Engineering_Practices_TOP.txt  requirements.txt
app.py                                  temp/
app1.py                                 ~$tching_a_story.docx
app2.py


In [13]:
# enter the full path of the input document
input_file_path = '/Users/alexanderarefolov/Dropbox/Coding_Projects/documents/Amendment_One_Master_Deed.pdf'

input_file_name = os.path.basename(input_file_path)

In [14]:
input_file_name

'Amendment_One_Master_Deed.pdf'

In [15]:
# S3 bucket name is hardcoded into environment variables
bucket_name = os.getenv("BUCKET_NAME")

In [16]:
bucket_name

'real.estate.rag-user.sessions.docs-test-115356603380-useast2'

In [17]:
# enter the name of the session.  This name is used for construction of S3 key together with file name
# and it is used to name the namespace in pinecone index
namespace = 'test_user1_textract'

In [18]:
question = "What is the document about?"

### Main Code

In [19]:
# create ASW S3 bucket if it does not alfready exist
create_bucket(bucket_name = bucket_name)

An error occurred (BucketAlreadyOwnedByYou) when calling the CreateBucket operation: Your previous request to create the named bucket succeeded and you already own it.


False

In [20]:
# upload my file into the bucket using my defined folder structure
upload_file_s3(file_name = input_file_path, bucket = bucket_name, namespace = namespace, s3_file_name = input_file_name)

file /Users/alexanderarefolov/Dropbox/Coding_Projects/documents/Amendment_One_Master_Deed.pdf has been uploaded into the bucket real.estate.rag-user.sessions.docs-test-115356603380-useast2.


True

In [21]:
s3_url = get_s3_url(
    bucket = bucket_name, 
    namespace = namespace, 
    s3_file_name = input_file_name
    )

s3_url

's3://real.estate.rag-user.sessions.docs-test-115356603380-useast2/test_user1_textract/Amendment_One_Master_Deed.pdf'

In [ ]:
pip list

In [ ]:
s3_url

In [22]:
# load documents via AmazonTextractPDFLoader
documents = load_document_AWS_textract(s3_url)

documents

Loaded documents 10 pages long


[Document(metadata={'source': 's3://real.estate.rag-user.sessions.docs-test-115356603380-useast2/test_user1_textract/Amendment_One_Master_Deed.pdf', 'page': 1}, page_content='AMENDMENT NUMBER ONE TO THE\n\n\nMASTER DEED OF THE\n\n\nINDEPENDENCE CONDOMINIUM\n\n\nReference is hereby made to that certain Master Deed dated\n\n\nAugust 12, 1981, and recorded with the Norfolk County Registry\n\n\nof Deeds in Book 5909, Page 203, which Master Deed established,\n\n\npursuant to Massachusetts General Laws, Chapter 183A, the\n\n\nIndependence Condominium.\n\n\nWHEREAS the Unit Owners entitled to more than seventy-five\n\n\npercent (75%) of the Undivided Interest desire to amend said\n\n\nMaster Deed as provided in Paragraph 8 thereof.\n\n\nWHEREAS no other consents are required therefor.\n\n\nNOW THEREFORE said Master Deed is hereby amended in\n\n\naccordance with the provisions of Paragraph 8 of said Master\n\n\nDeed as follows:\n\n\n1. Paragraph 7.1 is amended by adding at the end thereof\n\n\

In [32]:
textract_client = boto3.client("textract", region_name="us-east-2")

In [33]:
loader = AmazonTextractPDFLoader(file_path = s3_url, client=textract_client)

In [ ]:
loader

In [ ]:
documents = loader.load()

print(f"Loaded documents {len(documents)} pages long")

In [28]:
# chunk the document
chunks = chunk_data(documents, chunk_size = 512)
print(f"Chunk size: 512, Chunks: {len(chunks)}")

Chunk size: 512, Chunks: 45


In [29]:
tokens, embedding_cost = calculate_input_embedding_cost(chunks)
print(f'Source document embedding cost: ${embedding_cost:.4f}')

Source document embedding cost: $0.0001


In [30]:
# creating the embeddings and returning the Pinecone vector store
vector_store = create_embeddings(chunks = chunks, namespace = namespace)
print(vector_store)

Created vector store within real-estate-rag index and in test_user1_textract namespace


In [31]:
# build messages
system_template = r'''
You are answering questions only concerning the provided content of the input document.  
If you are asked a question that is not related to the document you response will be:
'I can answer only the questions related to the source document!'.
---------------
Context: ```{context}```
'''

user_template = '''
Answer questions only concerning the provided content of the input document.  
If you are asked a question that is not related to the document you response will be:
'I can answer only the questions related to the source document!'. 
Here is the user's question: ```{question}```
'''

messages= [
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(user_template)
    ]

qa_prompt = ChatPromptTemplate.from_messages(messages)

In [32]:
# initialize LLM
llm = ChatOpenAI(
    api_key = os.getenv("OPENAI_API_KEY"),  
    model = os.getenv("OPENAI_DEPLOYMENT_NAME"), 
    temperature = 0)

In [33]:
# Configure vector store to act as a retriever (finding similar items, returning top k)
retriever = vector_store.as_retriever(
    search_type='similarity', search_kwargs={'k': 3, 'namespace': namespace})

In [34]:
# Create a memory buffer to track the conversation
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

In [35]:
 # Set up conversational retrieval chain
crc = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = retriever,
    memory = memory,
    chain_type = 'stuff',
    combine_docs_chain_kwargs = {'prompt': qa_prompt },
    verbose = False)

In [36]:
result = crc.invoke({'question': question})
response = result['answer']

In [37]:
response

'The document is about an amendment to the Master Deed of the Independence Condominium, originally established on August 12, 1981. It mentions that the unit owners, who hold more than seventy-five percent of the Undivided Interest, desire to amend the Master Deed. The document also outlines conditions under which alterations to the building can be made, including the requirement for approval from the Trustees and the provision of plans and specifications for any proposed alterations.'